# 🎙️ VoiceForge Server

خادم API لتطبيق VoiceForge — بديل Applio للموبايل

## الخطوات:
1. تشغيل خلية **الإعداد والتثبيت**
2. تشغيل خلية **تشغيل الخادم**
3. انسخ الرابط الذي يظهر وألصقه في تطبيق VoiceForge على هاتفك

> ⚠️ **مهم:** اختر GPU من Runtime > Change runtime type > T4 GPU

In [ ]:
#@title 1️⃣ الإعداد والتثبيت { display-mode: "form" }
#@markdown ### تركيب Google Drive وتثبيت Applio + خادم VoiceForge
#@markdown ---
mount_drive = True #@param {type:"boolean"}

import os

# Mount Google Drive
if mount_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Google Drive mounted')

# Check GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = getattr(torch.cuda.get_device_properties(0), 'total_memory', 0) / 1024**3
    print(f'✅ GPU: {gpu} ({mem:.1f} GB)')
else:
    print('⚠️ لم يتم العثور على GPU! اذهب إلى Runtime > Change runtime type > T4 GPU')

# Clone Applio
if not os.path.exists('/content/Applio'):
    !git clone https://github.com/IAHispano/Applio.git /content/Applio
    %cd /content/Applio
    !pip install -q -r requirements.txt
    print('✅ Applio installed')
else:
    %cd /content/Applio
    print('✅ Applio already installed')

# Install server dependencies
!pip install -q fastapi uvicorn nest-asyncio python-multipart aiofiles psutil

# Install Cloudflare tunnel (free, no account needed)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print('\n✅ Setup complete!')

In [ ]:
#@title 2️⃣ تشغيل الخادم { display-mode: "form" }
#@markdown ### إعدادات الخادم
#@markdown ---
FIREBASE_DB_URL = "" #@param {type:"string"}
#@markdown رابط Firebase Realtime Database (اختياري — لربط تلقائي مع التطبيق)
#@markdown
#@markdown مثال: `https://your-project-default-rtdb.firebaseio.com`

import sys
import os
import json
import threading
import subprocess
import re
import time
import secrets
import nest_asyncio
import asyncio
import torch
import psutil
import uuid
import shutil
import glob
from typing import List, Optional

import uvicorn
from fastapi import FastAPI, WebSocket, WebSocketDisconnect, UploadFile, File, Form, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel

nest_asyncio.apply()
sys.path.insert(0, '/content/Applio')

# ─── Config ───
API_KEY = secrets.token_urlsafe(16)
PORT = 8000
APPLIO_DIR = '/content/Applio'
WEIGHTS_DIR = os.path.join(APPLIO_DIR, 'logs')
DATASETS_DIR = os.path.join(APPLIO_DIR, 'assets', 'datasets')
OUTPUT_DIR = '/content/outputs'
DRIVE_DIR = '/content/drive/MyDrive/VoiceForge/models'

for d in [OUTPUT_DIR, DATASETS_DIR, WEIGHTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ─── FastAPI App ───
app = FastAPI(title='VoiceForge API')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

# ─── WebSocket Manager ───
class ConnectionManager:
    def __init__(self):
        self.connections = []
    async def connect(self, ws):
        await ws.accept()
        self.connections.append(ws)
    def disconnect(self, ws):
        if ws in self.connections:
            self.connections.remove(ws)
    async def broadcast(self, msg):
        for ws in self.connections[:]:
            try:
                await ws.send_json(msg)
            except:
                self.connections.remove(ws)

manager = ConnectionManager()
app.state.ws_manager = manager

@app.websocket('/ws')
async def ws_endpoint(ws: WebSocket):
    await manager.connect(ws)
    try:
        while True:
            await ws.receive_text()
    except WebSocketDisconnect:
        manager.disconnect(ws)

# ─── Status ───
@app.get('/api/status')
async def get_status():
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'
    return {
        'status': 'online', 'gpu': gpu,
        'cuda_available': torch.cuda.is_available(),
        'applio_installed': os.path.exists(os.path.join(APPLIO_DIR, 'core.py')),
        'ram_used': f'{psutil.virtual_memory().used/1024**3:.1f} GB',
        'disk_free': f'{psutil.disk_usage("/").free/1024**3:.1f} GB',
    }

# ─── Models ───
class DownloadRequest(BaseModel):
    url: str
    name: Optional[str] = None

@app.get('/api/models')
async def list_models():
    models = []
    if os.path.exists(WEIGHTS_DIR):
        for entry in os.listdir(WEIGHTS_DIR):
            d = os.path.join(WEIGHTS_DIR, entry)
            if not os.path.isdir(d): continue
            pths = glob.glob(os.path.join(d, '*.pth'))
            idxs = glob.glob(os.path.join(d, '*.index'))
            if pths:
                models.append({'name': entry, 'pth_path': pths[0], 'index_path': idxs[0] if idxs else None,
                               'size_mb': round(os.path.getsize(pths[0])/1024/1024, 1)})
    return {'models': models}

@app.post('/api/models/upload')
async def upload_model(files: List[UploadFile] = File(...)):
    saved = []
    for f in files:
        dest = os.path.join(WEIGHTS_DIR, f.filename)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        with open(dest, 'wb') as out:
            out.write(await f.read())
        saved.append(f.filename)
    return {'message': f'Uploaded {len(saved)} files', 'files': saved}

@app.post('/api/models/download')
async def download_model(req: DownloadRequest):
    name = req.name or 'downloaded_model'
    dest = os.path.join(WEIGHTS_DIR, name)
    os.makedirs(dest, exist_ok=True)
    subprocess.run(['wget', '-q', req.url, '-P', dest], check=True)
    return {'message': f'Downloaded to {dest}'}

@app.delete('/api/models/{name}')
async def delete_model(name: str):
    d = os.path.join(WEIGHTS_DIR, name)
    if os.path.isdir(d):
        shutil.rmtree(d)
        return {'message': f'Deleted {name}'}
    raise HTTPException(404, 'Model not found')

# ─── Inference ───
@app.post('/api/inference')
async def run_inference(
    audio: UploadFile = File(...),
    pth_path: str = Form(...),
    index_path: str = Form(''),
    pitch: int = Form(0),
    f0_method: str = Form('rmvpe'),
    index_rate: float = Form(0.5),
    filter_radius: int = Form(3),
    volume_envelope: float = Form(0.25),
    protect_voiceless: float = Form(0.33),
):
    ext = os.path.splitext(audio.filename)[1] or '.wav'
    inp = os.path.join(OUTPUT_DIR, f'in_{uuid.uuid4().hex}{ext}')
    out = os.path.join(OUTPUT_DIR, f'out_{uuid.uuid4().hex}.wav')
    with open(inp, 'wb') as f:
        f.write(await audio.read())

    cmd = [sys.executable, os.path.join(APPLIO_DIR, 'core.py'), 'infer',
           '--input-path', inp, '--output-path', out,
           '--pth-path', pth_path, '--pitch', str(pitch),
           '--f0-method', f0_method, '--index-rate', str(index_rate),
           '--filter-radius', str(filter_radius),
           '--volume-envelope', str(volume_envelope),
           '--protect', str(protect_voiceless)]
    if index_path:
        cmd.extend(['--index-path', index_path])

    r = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    os.remove(inp) if os.path.exists(inp) else None

    if r.returncode != 0:
        raise HTTPException(500, f'Inference failed: {r.stderr[-300:]}')
    if not os.path.exists(out):
        raise HTTPException(500, 'No output generated')
    return FileResponse(out, media_type='audio/wav', filename=f'voiceforge_{uuid.uuid4().hex[:8]}.wav')

# ─── Training ───
class PreprocessCfg(BaseModel):
    model_name: str
    sample_rate: int = 40000

class ExtractCfg(BaseModel):
    model_name: str
    f0_method: str = 'rmvpe'

class TrainCfg(BaseModel):
    model_name: str
    sample_rate: str = '40000'
    f0_method: str = 'rmvpe'
    epochs: int = 100
    batch_size: int = 8
    save_frequency: int = 25

train_state = {'active': False, 'process': None, 'status': 'idle',
               'epoch': 0, 'total_epochs': 0, 'loss_g': 0.0, 'loss_d': 0.0, 'message': ''}

@app.post('/api/training/upload-dataset')
async def upload_dataset(model_name: str = Form(...), files: List[UploadFile] = File(...)):
    d = os.path.join(DATASETS_DIR, model_name)
    os.makedirs(d, exist_ok=True)
    saved = []
    for f in files:
        with open(os.path.join(d, f.filename), 'wb') as out:
            out.write(await f.read())
        saved.append(f.filename)
    return {'message': f'Uploaded {len(saved)} files', 'dataset_dir': d}

@app.post('/api/training/preprocess')
async def preprocess(cfg: PreprocessCfg):
    cmd = [sys.executable, os.path.join(APPLIO_DIR, 'core.py'), 'preprocess',
           '--model-name', cfg.model_name, '--sample-rate', str(cfg.sample_rate),
           '--dataset-path', os.path.join(DATASETS_DIR, cfg.model_name)]
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
    if r.returncode != 0:
        raise HTTPException(500, f'Failed: {r.stderr[-300:]}')
    return {'message': 'Preprocessing done'}

@app.post('/api/training/extract-features')
async def extract_features(cfg: ExtractCfg):
    cmd = [sys.executable, os.path.join(APPLIO_DIR, 'core.py'), 'extract',
           '--model-name', cfg.model_name, '--f0-method', cfg.f0_method]
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=1200)
    if r.returncode != 0:
        raise HTTPException(500, f'Failed: {r.stderr[-300:]}')
    return {'message': 'Feature extraction done'}

@app.post('/api/training/start')
async def start_training(cfg: TrainCfg, request: Request):
    global train_state
    if train_state['active']:
        raise HTTPException(409, 'Training already running')

    train_state.update(active=True, status='training', total_epochs=cfg.epochs, epoch=0)
    ws = request.app.state.ws_manager

    def _run():
        global train_state
        try:
            cmd = [sys.executable, os.path.join(APPLIO_DIR, 'core.py'), 'train',
                   '--model-name', cfg.model_name, '--sample-rate', cfg.sample_rate,
                   '--epochs', str(cfg.epochs), '--batch-size', str(cfg.batch_size),
                   '--save-every', str(cfg.save_frequency)]
            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            train_state['process'] = proc

            for line in proc.stdout:
                line = line.strip()
                if 'epoch' in line.lower():
                    try:
                        import re as _re
                        m = _re.search(r'(\d+)\s*/\s*(\d+)', line)
                        if m:
                            train_state['epoch'] = int(m.group(1))
                            train_state['total_epochs'] = int(m.group(2))
                    except: pass
                train_state['message'] = line[-100:]
                pct = (train_state['epoch'] / max(train_state['total_epochs'], 1)) * 100
                try:
                    loop = asyncio.get_event_loop()
                    asyncio.ensure_future(ws.broadcast({
                        'type': 'training_progress',
                        'payload': {'epoch': train_state['epoch'], 'totalEpochs': train_state['total_epochs'],
                                    'loss_g': train_state['loss_g'], 'loss_d': train_state['loss_d'],
                                    'status': 'training', 'message': train_state['message'], 'percentage': pct}
                    }))
                except: pass

            proc.wait()
            train_state['status'] = 'completed' if proc.returncode == 0 else 'error'
        except Exception as e:
            train_state['status'] = 'error'
            train_state['message'] = str(e)
        finally:
            train_state['active'] = False
            train_state['process'] = None

    threading.Thread(target=_run, daemon=True).start()
    return {'message': 'Training started'}

@app.post('/api/training/stop')
async def stop_training():
    if train_state.get('process'):
        train_state['process'].terminate()
        train_state['active'] = False
        train_state['status'] = 'idle'
        return {'message': 'Stopped'}
    raise HTTPException(404, 'No active training')

@app.get('/api/training/status')
async def training_status():
    return {k: v for k, v in train_state.items() if k != 'process'}

# ─── Drive Sync ───
class SyncReq(BaseModel):
    model_name: str

@app.post('/api/drive/sync')
async def sync_drive(req: SyncReq):
    if not os.path.exists('/content/drive/MyDrive'):
        raise HTTPException(400, 'Drive not mounted')
    os.makedirs(DRIVE_DIR, exist_ok=True)
    src = os.path.join(WEIGHTS_DIR, req.model_name)
    if os.path.isdir(src):
        dest = os.path.join(DRIVE_DIR, req.model_name)
        if os.path.exists(dest): shutil.rmtree(dest)
        shutil.copytree(src, dest)
        return {'message': f'Synced to {dest}'}
    raise HTTPException(404, 'Model not found')

# ─── Start Server ───
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=PORT)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)

# ─── Cloudflare Tunnel (free, no account needed) ───
process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

tunnel_url = None
for line in process.stdout:
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

# Update Firebase if configured
if tunnel_url and FIREBASE_DB_URL:
    try:
        import requests as _req
        _req.put(f'{FIREBASE_DB_URL}/voiceforge/serverUrl.json', json=tunnel_url)
        print(f'✅ Firebase updated with tunnel URL')
    except Exception as e:
        print(f'⚠️ Firebase update failed: {e}')

print(f'''
{'='*60}
  🎙️  VoiceForge Server is running!
{'='*60}

  📱 Server URL (paste in app):
  👉 {tunnel_url}

  🔑 API Key (optional):
  👉 {API_KEY}

  📋 API Docs:
  👉 {tunnel_url}/docs

{'='*60}
''')

# Keep reading tunnel output in background
def _keep_tunnel():
    for line in process.stdout:
        pass
threading.Thread(target=_keep_tunnel, daemon=True).start()

In [ ]:
#@title 3️⃣ إبقاء الجلسة نشطة (Keep Alive) { display-mode: "form" }
#@markdown شغّل هذه الخلية لمنع Colab من قطع الاتصال بسبب عدم النشاط

import IPython
from IPython.display import display, HTML

display(HTML('''
<script>
  let keepAliveCount = 0;
  function keepAlive() {
    keepAliveCount++;
    fetch('/').then(() => {
      document.getElementById('ka-status').innerText = 
        '✅ نشط | Ping #' + keepAliveCount + ' | ' + new Date().toLocaleTimeString('ar-EG');
    }).catch(() => {
      document.getElementById('ka-status').innerText = '⚠️ فشل الـ Ping';
    });
  }
  setInterval(keepAlive, 60000);
  keepAlive();
</script>
<div style="padding: 10px; background: #1a1a2e; border-radius: 8px; color: #e0e0e0; font-family: monospace;">
  <span>🔄 Keep Alive: </span><span id="ka-status">جاري التفعيل...</span>
</div>
'''))

print('✅ Keep-alive activated — ping every 60 seconds')

In [ ]:
#@title 🔄 إعادة عرض رابط الخادم { display-mode: "form" }
#@markdown إذا ضاع الرابط، شغّل هذه الخلية لإعادة عرضه

if tunnel_url:
    print(f'📱 Server URL: {tunnel_url}')
    print(f'🔑 API Key: {API_KEY}')
    print(f'📋 API Docs: {tunnel_url}/docs')
else:
    print('⚠️ لم يتم العثور على رابط. أعد تشغيل خلية الخادم.')